In [ ]:
"""
═══════════════════════════════════════════════════════════════════════════════
MNIST QPSK Neural Network with QPSK Inputs, Activations, and Outputs
Real-Valued Weights/Biases with Phase-Aligned Quantization-Aware Training
Final Test Accuracy: 92%
═══════════════════════════════════════════════════════════════════════════════

This project implements a complex-valued QPSK neural network for MNIST
classification using phase-aligned quantization-aware training (QAT) with
Straight Through Estimator (STE). Input pixels are converted into strict
QPSK symbols and propagated through complex-valued fully connected layers
with hard QPSK activation quantization applied after each hidden layer.
Unlike fully quantized models, the network retains real-valued weights and
biases during training while enforcing discrete QPSK activations and outputs.
The model is trained using MSE loss on complex pre-activations against
explicit QPSK constellation targets, allowing simultaneous optimization of
both phase and magnitude instead of relying only on softmax probabilities.
Inference is performed entirely using strict QPSK OFDM symbols without soft
logits, making the network communication-friendly and hardware-efficient.
The final model achieves approximately 92% MNIST classification accuracy
while maintaining fully discrete QPSK input, hidden activation, and output
representations suitable for OFDM transmission systems.
"""

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. QPSK Activation with STE

# Custom autograd function that hard-quantizes complex activations to the QPSK
# constellation in the forward pass: each component's sign is taken (zeros
# default to +1) and scaled by 1/√2, placing every output exactly on one of
# the four QPSK points. The backward pass uses a clipped STE, zeroing gradients
# for elements whose magnitude exceeds 1.0 to prevent updates from saturated
# activations that lie well beyond the constellation range.
class QPSKActivation(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input_tensor):
        ctx.save_for_backward(input_tensor)
        real_sign = torch.sign(input_tensor.real)
        real_sign[real_sign == 0] = 1.0
        imag_sign = torch.sign(input_tensor.imag)
        imag_sign[imag_sign == 0] = 1.0

        scale = 1.0 / 1.41421356
        return torch.complex(real_sign * scale, imag_sign * scale)

    @staticmethod
    def backward(ctx, grad_output):
        input_tensor, = ctx.saved_tensors
        grad_input_real = grad_output.real.clone()
        grad_input_real[torch.abs(input_tensor.real) > 1.0] = 0.0
        grad_input_imag = grad_output.imag.clone()
        grad_input_imag[torch.abs(input_tensor.imag) > 1.0] = 0.0
        return torch.complex(grad_input_real, grad_input_imag)

# 2. Complex Linear Layer

# A complex-valued fully-connected layer implemented using two standard real
# nn.Linear modules (fc_r for the real weight matrix, fc_i for the imaginary
# weight matrix). The complex matrix-vector product (a+jb)(c+jd) is expanded
# into four real operations: real output = xr·wr − xi·wi, imaginary output =
# xr·wi + xi·wr. This avoids needing native complex CUDA kernels while
# correctly modelling complex multiplication including bias terms from each sub-layer.
class ComplexLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super(ComplexLinear, self).__init__()
        self.fc_r = nn.Linear(in_features, out_features)
        self.fc_i = nn.Linear(in_features, out_features)

    # Computes the complex linear transformation by splitting x into real and
    # imaginary parts and applying the four-real-matmul expansion, then
    # reassembling the result as a complex tensor.
    def forward(self, x):
        real_out = self.fc_r(x.real) - self.fc_i(x.imag)
        imag_out = self.fc_r(x.imag) + self.fc_i(x.real)
        return torch.complex(real_out, imag_out)

# 3. Sound QAT Network

# Three-layer complex MLP for MNIST trained with phase-aligned quantization-aware
# training (QAT). Inputs are hard-quantized to QPSK symbols at entry (alternating
# pixels → real/imag, sign-binarized and scaled). Hidden layers apply ComplexLinear
# followed by strict QPSK quantization. The output layer returns three tensors:
# the raw complex pre-activation (used for MSE loss during training), its magnitude
# (available as logits for cross-entropy if needed), and the hard-quantized QPSK
# symbols ready for OFDM transmission.
class SoundQPSKNet(nn.Module):
    def __init__(self):
        super(SoundQPSKNet, self).__init__()
        self.fc1 = ComplexLinear(392, 512)
        self.fc2 = ComplexLinear(512, 512)
        self.fc3 = ComplexLinear(512, 10)
        self.qpsk = QPSKActivation.apply

    # Converts a float image batch into QPSK-constrained complex inputs,
    # passes them through two quantized hidden layers, then returns three
    # views of the output layer: the raw complex pre-activation for MSE
    # loss, its element-wise magnitude for classification, and the hard-
    # quantized QPSK symbols for OFDM use.
    def forward(self, x):
        x = x.view(x.size(0), -1)

        # Strict QPSK Input Generation
        x_bin = torch.sign(x)
        x_bin[x_bin == 0] = 1.0
        scale = 1.0 / 1.41421356
        x_c = torch.complex(x_bin[:, 0::2] * scale, x_bin[:, 1::2] * scale)

        # Hidden Layers (Strict QPSK)
        x_c = self.qpsk(self.fc1(x_c))
        x_c = self.qpsk(self.fc2(x_c))

        # Output Layer Pre-activation (this is the complex value needed for MSE)
        complex_pre_activation = self.fc3(x_c)

        # Continuous output for training loss (magnitude for CrossEntropyLoss if needed)
        train_logits = torch.abs(complex_pre_activation)

        # Strict QPSK output for OFDM block
        ofdm_symbols = self.qpsk(complex_pre_activation)

        # Return the complex pre-activation, train_logits, and ofdm_symbols
        return complex_pre_activation, train_logits, ofdm_symbols

# 4. Strict QPSK Evaluation Function

# Evaluates classification accuracy using only the hard-quantized QPSK output
# symbols (the third return value from the model), making no use of soft logits
# or magnitudes. Classification is done by finding the output class whose QPSK
# symbol is closest (in complex magnitude) to the Quadrant-1 target point
# (+1/√2 + j/√2), which is the designated "correct class" symbol during training.
def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0

    scale = 1.0 / 1.41421356
    target_symbol = torch.complex(torch.tensor(scale), torch.tensor(scale)).to(device)

    with torch.no_grad(): # Disables gradient calculations entirely
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            # Extract ONLY the strictly discrete QPSK symbols (ofdm_symbols is the 3rd return value)
            _, _, ofdm_symbols = model(images)

            # Classification based on minimum distance to target symbol
            distances = torch.abs(ofdm_symbols - target_symbol)
            _, predicted = torch.min(distances, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Final Test Accuracy (100% QPSK Inference): {accuracy:.2f}%")
    return accuracy


# Builds a batch of complex target tensors for MSE-based QAT training.
# Each target is a (batch_size, num_classes) complex matrix initialized to
# Quadrant-3 (−1/√2 − j/√2) for all classes, with the ground-truth class
# position overwritten to Quadrant-1 (+1/√2 + j/√2). Minimizing MSE between
# the model's complex pre-activation and these targets simultaneously optimizes
# both magnitude and phase, driving the correct class output toward the
# designated QPSK symbol while suppressing all others.
def get_qpsk_targets(labels, num_classes=10, device='cpu'):
    """Generates explicit QPSK coordinate targets for MSE Loss."""
    scale = 1.0 / 1.41421356
    # Initialize all to Quadrant 3 (-1 - j)
    targets = torch.complex(torch.full((labels.size(0), num_classes), -scale),
                            torch.full((labels.size(0), num_classes), -scale)).to(device)

    # Set correct class to Quadrant 1 (+1 + j)
    for i, label in enumerate(labels):
        targets[i, label] = torch.complex(torch.tensor(scale), torch.tensor(scale))

    return targets


# Entry point that sets up MNIST loaders, instantiates SoundQPSKNet, and trains
# for 10 epochs using MSE loss on complex pre-activations against QPSK targets
# (rather than cross-entropy on magnitudes) so that both the phase and magnitude
# of each output neuron are explicitly optimized toward valid QPSK points. Training
# accuracy is measured by minimum-distance decoding against the Quadrant-1 target.
# After training, calls evaluate_model() for a strict QPSK-only inference pass.
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

    test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

    model = SoundQPSKNet().to(device)

    # FIX: Use MSE Loss to optimize both magnitude AND phase
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

    epochs = 10
    print("--- Starting Training (Phase-Aligned QAT) ---")
    for epoch in range(epochs):
        model.train()
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # Capture complex_pre_activation (the actual complex output of fc3)
            complex_pre_activation, train_logits, ofdm_symbols = model(images)

            targets = get_qpsk_targets(labels, device=device)

            # MSE requires real-valued tensors, so we view the complex tensors as real (adds a dimension for Re/Im)
            loss = criterion(torch.view_as_real(complex_pre_activation), torch.view_as_real(targets))
            loss.backward()
            optimizer.step()

            # Calculate training accuracy based on distance to the target Quadrant 1 point
            scale = 1.0 / 1.41421356
            target_symbol = torch.complex(torch.tensor(scale), torch.tensor(scale)).to(device)
            distances = torch.abs(complex_pre_activation - target_symbol) # Use complex_pre_activation
            _, predicted = torch.min(distances, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f"Epoch {epoch+1}/{epochs} - Train Accuracy: {100 * correct / total:.2f}%")

    print("\n--- Starting Strict QPSK Testing ---")
    evaluate_model(model, test_loader, device)

if __name__ == "__main__":
    main()